<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.2-few-shot-and-in-context/notebooks/exercises-3_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.2: Few-Shot and In-Context Learning

*Module 3 · Prompt Engineering & Context Design*

> You don't need to re-train a model to teach it a new task. Just show it 3 examples and it figures out the pattern. Let's learn how — and how to pick the BEST examples automatically.

---

## About this notebook

This notebook contains the **10 runnable code examples** from the published lesson `lesson-3.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Zero-Shot vs Few-Shot — See the Difference — `part1_zero_vs_few.py`
2. Step 2 — Building Few-Shot Prompts — The Right Format — `part2_few_shot_builder.py`
3. Step 3 — Why In-Context Learning Works — The Science Behind Examples — `icl_science.py`
4. Step 4 — How Many Examples? — Finding the Sweet Spot — `part3_example_count.py`
5. Step 5 — Ordering Effects — Yes, the Order Matters — `part4_order_matters.py`
6. Step 6 — Smart Example Selection — Pick the Best Examples Automatically — `part5_embedding_selection.py`
7. Step 7 — Many-Shot Prompting — 100+ Examples in 2M Token Contexts — `many_shot.py`
8. Step 8 — Contrastive Examples & When Few-Shot Hurts — `contrastive.py`
9. Step 9 — Prompt Caching & Token Budgets — Make Examples Nearly Free — `caching.py`
10. Step 10 — Project: Dynamic Few-Shot Classification System — `project_dynamic_fewshot.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy requests datasets


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Zero-Shot vs Few-Shot — See the Difference

Does showing examples actually help? Let's prove it with an API test.

**`part1_zero_vs_few.py`** · _Block 1 of 10_

EXPERIMENT: Zero-shot vs Few-shot — Task: Classify customer support emails


In [ ]:
# =============================================
# EXPERIMENT: Zero-shot vs Few-shot
# Task: Classify customer support emails
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def ask(prompt, temp=0.1):
    model = genai.GenerativeModel("gemini-2.0-flash", generation_config={"temperature": temp})
    return model.generate_content(prompt).text.strip()

test_email = "My laptop won't connect to the company VPN since the last update."

# ── ZERO-SHOT: No examples, just instructions ──
zero_shot = f"""Classify this customer email into exactly one category: 
Hardware, Software, Billing, Connectivity, or Account.
Reply with ONLY the category name, nothing else.

Email: "{test_email}"
Category:"""

# ── FEW-SHOT: Show 3 examples first ──
few_shot = f"""Classify customer emails into exactly one category.

Email: "My monitor keeps flickering and showing weird colors."
Category: Hardware

Email: "I was charged twice for my subscription this month."
Category: Billing

Email: "The new app update crashes every time I open it."
Category: Software

Email: "{test_email}"
Category:"""

print("Task: Classify a VPN connection email\n")

zero_result = ask(zero_shot)
few_result = ask(few_shot)

print(f"  Zero-shot answer: {zero_result}")
print(f"  Few-shot answer:  {few_result}")
print(f"  Expected:         Connectivity (or Software)")

# Zero-shot might say "Software" or "Network" or "IT Support"
# Few-shot almost always says "Connectivity" — it learned the pattern!


> **What just happened?** Zero-shot gave a reasonable but inconsistent answer — sometimes "Software", sometimes "Network" (which isn't even in our categories). Few-shot saw the pattern from 3 examples and gave a consistent answer from our exact category list. Examples teach the model your specific format, categories, and thinking pattern.


### Step 2 · Building Few-Shot Prompts — The Right Format

Structure matters. A well-formatted few-shot prompt works 2x better than a messy one.

**`part2_few_shot_builder.py`** · _Block 2 of 10_

BUILD A REUSABLE FEW-SHOT PROMPT SYSTEM — Works for any classification task — just


In [ ]:
# =============================================
# BUILD A REUSABLE FEW-SHOT PROMPT SYSTEM
# Works for any classification task — just
# change the examples!
# =============================================

def build_few_shot_prompt(
    task_description: str,
    examples: list[tuple[str, str]],  # [(input, output), ...]
    query: str,
    input_label: str = "Input",
    output_label: str = "Output",
) -> str:
    """Build a clean few-shot prompt from examples."""
    
    # Start with the task description
    prompt = task_description.strip() + "\n\n"
    
    # Add each example in a consistent format
    for inp, out in examples:
        prompt += f"{input_label}: {inp}\n{output_label}: {out}\n\n"
    
    # Add the actual query (no answer — the model fills this in)
    prompt += f"{input_label}: {query}\n{output_label}:"
    
    return prompt

# ── EXAMPLE 1: Sentiment Analysis ──
sentiment_examples = [
    ("This product is absolutely amazing! Best purchase ever.", "positive"),
    ("Terrible quality. Broke after 2 days. Want a refund.", "negative"),
    ("It's okay. Nothing special but works fine for the price.", "neutral"),
    ("Love the design but the battery dies too fast.", "mixed"),
]

prompt = build_few_shot_prompt(
    task_description="Classify the sentiment of product reviews. Reply with exactly one word: positive, negative, neutral, or mixed.",
    examples=sentiment_examples,
    query="Great camera quality but terrible customer service when I needed help.",
    input_label="Review",
    output_label="Sentiment",
)

print("═══ PROMPT SENT TO AI ═══")
print(prompt)
print("\n═══ AI RESPONSE ═══")
print(ask(prompt))
# → "mixed" (positive about camera, negative about service)

# ── EXAMPLE 2: Language Detection ──
lang_examples = [
    ("Bonjour, comment allez-vous?", "French"),
    ("Wie geht es Ihnen?", "German"),
    ("Namaste, aap kaise hain?", "Hindi"),
    ("Hola, como estas?", "Spanish"),
]

prompt2 = build_few_shot_prompt(
    task_description="Identify the language of the text. Reply with the language name only.",
    examples=lang_examples,
    query="Mujhe yeh bahut pasand aaya!",
    input_label="Text",
    output_label="Language",
)

print(f"\nLanguage detection: {ask(prompt2)}")
# → "Hindi" (Hinglish to be precise)


> **What just happened?** We built a reusable build_few_shot_prompt() function that takes any task description, any set of examples, and any query — and produces a clean, consistently formatted prompt. We tested it on two different tasks (sentiment and language detection) without changing the function. The AI learned the pattern from 4 examples each time. This is the core building block of few-shot learning.


### Step 3 · Why In-Context Learning Works — The Science Behind Examples

Min et al. 2022 changed everything: correct labels barely matter. Format is king.

**`icl_science.py`** · _Block 3 of 10_

MIN ET AL. 2022 — "Rethinking Demonstrations" — Tested on 12 models + 26 datasets (incl. GPT-3)


In [ ]:
# =============================================
# MIN ET AL. 2022 — "Rethinking Demonstrations"
# Tested on 12 models + 26 datasets (incl. GPT-3)
# =============================================

print("""
WHAT MATTERS IN FEW-SHOT (ranked by importance):
  1. FORMAT / STRUCTURE  → Critical! Consistent template is #1
  2. LABEL SPACE         → Must show all possible labels
  3. INPUT DISTRIBUTION  → Examples from same domain
  4. CORRECT LABELS      → Barely matters! Random labels ≈ correct labels

  SHOCKING FINDING:
    Replacing correct labels with RANDOM labels
    barely hurt performance across 26 datasets!
    
    → Examples teach PATTERN, not knowledge
    → The model recognizes the TASK from format
    → Correct labels help more in larger models (GPT-4o+)

THE THREE BIASES (Zhao et al. 2021):
  1. MAJORITY LABEL BIAS:  3 positive + 1 negative → model over-predicts positive
     FIX: Balance your example classes equally
     
  2. RECENCY BIAS:  Last example = strongest influence
     Reversing 2 examples on SST-2: accuracy 88.5% → 51.3%!
     FIX: Put most relevant example LAST
     
  3. COMMON TOKEN BIAS:  Model prefers frequent words
     FIX: Use calibration (content-free input to measure bias)
""")


> **What just happened?** The Min et al. 2022 paper (EMNLP) proved that format and label space matter far more than correct labels in few-shot prompting. Your examples primarily serve as task specification , not training data. Three biases (majority label, recency, common token) explain why few-shot is so sensitive to example selection and ordering. For production: balance classes, put strongest example last, use consistent format.


### Step 4 · How Many Examples? — Finding the Sweet Spot

More examples isn't always better. Let's measure it.

**`part3_example_count.py`** · _Block 4 of 10_

EXPERIMENT: How many examples do you need? — Test 0, 1, 3, 5, and 10 examples on the


In [ ]:
# =============================================
# EXPERIMENT: How many examples do you need?
# Test 0, 1, 3, 5, and 10 examples on the
# same task and measure accuracy.
# =============================================

import json

# Task: Classify support tickets into categories
all_examples = [
    ("My screen is cracked", "Hardware"),
    ("App crashes on startup", "Software"),
    ("Can't connect to WiFi", "Connectivity"),
    ("Charged wrong amount", "Billing"),
    ("Forgot my password", "Account"),
    ("Battery drains in 2 hours", "Hardware"),
    ("Update deleted my files", "Software"),
    ("Bluetooth keeps disconnecting", "Connectivity"),
    ("Refund not received", "Billing"),
    ("Change my email address", "Account"),
]

# Test tickets (we know the right answers)
test_cases = [
    ("Keyboard keys are stuck", "Hardware"),
    ("VPN connection drops every 5 minutes", "Connectivity"),
    ("Got charged for a cancelled subscription", "Billing"),
    ("Software freezes when opening large files", "Software"),
    ("Need to reset my two-factor authentication", "Account"),
]

task_desc = "Classify support tickets. Categories: Hardware, Software, Connectivity, Billing, Account. Reply with ONLY the category."

print("Testing accuracy with different numbers of examples:\n")

for num_examples in [0, 1, 3, 5, 10]:
    examples = all_examples[:num_examples]
    correct = 0
    
    for test_input, expected in test_cases:
        prompt = build_few_shot_prompt(task_desc, examples, test_input, "Ticket", "Category")
        result = ask(prompt).strip()
        if expected.lower() in result.lower():
            correct += 1
    
    accuracy = correct / len(test_cases) * 100
    bar = "#" * int(accuracy / 5)
    print(f"  {num_examples:2d} examples → {accuracy:5.0f}% accuracy  {bar}")

# Typical results:
#   0 examples →  60% (zero-shot: guesses some wrong)
#   1 example  →  60% (one example isn't enough pattern)
#   3 examples →  80% (starting to see the pattern!)
#   5 examples → 100% (sweet spot!)
#  10 examples → 100% (no improvement, wastes tokens)


> **What just happened?** We tested 0, 1, 3, 5, and 10 examples on the same 5 test cases. Accuracy jumped significantly from 0 to 3 examples, then hit near-perfect at 5. Adding more examples to 10 didn't help but used more tokens (= more cost). The sweet spot is usually 3-5 examples. Enough to show the pattern, not so many that you waste context window space.


### Step 5 · Ordering Effects — Yes, the Order Matters

Shuffling the same examples can change the accuracy by 20%. Seriously.

**`part4_order_matters.py`** · _Block 5 of 10_

EXPERIMENT: Does example ORDER matter? — Same examples, different orders.


In [ ]:
# =============================================
# EXPERIMENT: Does example ORDER matter?
# Same examples, different orders.
# Spoiler: YES, it matters a lot.
# =============================================

import random

examples_sentiment = [
    ("Amazing product, love it!", "positive"),
    ("Worst purchase ever, total waste.", "negative"),
    ("Works fine, nothing special.", "neutral"),
    ("Great features but poor battery.", "mixed"),
]

# This review is genuinely ambiguous (could be mixed or neutral)
ambiguous_review = "The camera is decent and the price is fair, but I expected better build quality."

task = "Classify the review sentiment. Reply with ONLY: positive, negative, neutral, or mixed."

print("Same examples, 5 different orderings:\n")
print(f"Review: \"{ambiguous_review[:60]}...\"\n")

results = []
for i in range(5):
    # Shuffle examples
    shuffled = examples_sentiment.copy()
    random.shuffle(shuffled)
    
    prompt = build_few_shot_prompt(task, shuffled, ambiguous_review, "Review", "Sentiment")
    result = ask(prompt).strip().lower()
    results.append(result)
    
    # Show what the LAST example was (most influential)
    last_sentiment = shuffled[-1][1]
    print(f"  Order {i+1}: last example was '{last_sentiment:8s}' → AI said: '{result}'")

print(f"\nAll answers: {results}")
print(f"Unique answers: {set(results)}")

if len(set(results)) > 1:
    print("\n⚠️ Different orderings gave DIFFERENT answers!")
    print("   This is the ordering effect — the last example biases the model.")
else:
    print("\n✅ This model was robust to ordering (but many aren't!)")


> **What just happened?** Same 4 examples, same review, but shuffled 5 times. For ambiguous inputs, the model often gives different answers depending on which example was last. This is called recency bias — the model is influenced most by what it saw most recently. Fix: put a diverse, balanced mix of categories at the end, or — even better — use the smart selection technique from Part 5.


### Step 6 · Smart Example Selection — Pick the Best Examples Automatically

Don't use random examples. Use the ones MOST SIMILAR to the current query.

**`part5_embedding_selection.py`** · _Block 6 of 10_

DYNAMIC EXAMPLE SELECTION WITH EMBEDDINGS — For each new query, find the 3 most similar


In [ ]:
# =============================================
# DYNAMIC EXAMPLE SELECTION WITH EMBEDDINGS
# For each new query, find the 3 most similar
# examples from our pool using cosine similarity.
# =============================================

import google.generativeai as genai
import numpy as np

# ── Step 1: Our pool of labeled examples ──
example_pool = [
    ("My phone screen is cracked and won't respond to touch", "Hardware"),
    ("The app keeps crashing whenever I try to upload a photo", "Software"),
    ("WiFi keeps dropping every few minutes on my laptop", "Connectivity"),
    ("I was double-charged for my monthly subscription", "Billing"),
    ("How do I reset my account password?", "Account"),
    ("The laptop battery only lasts 1 hour after charging", "Hardware"),
    ("Software update broke the dark mode feature", "Software"),
    ("Bluetooth headphones won't pair with my device", "Connectivity"),
    ("I need a refund for an accidental purchase", "Billing"),
    ("Can I change the email associated with my account?", "Account"),
    ("The trackpad clicks but doesn't register movement", "Hardware"),
    ("Video calls freeze and have no audio", "Connectivity"),
    ("The export feature generates corrupted PDF files", "Software"),
    ("My promo code isn't working at checkout", "Billing"),
    ("How do I enable two-factor authentication?", "Account"),
]

# ── Step 2: Embed all examples (do this ONCE and cache) ──
def embed(text):
    result = genai.embed_content(model="models/text-embedding-004", content=text, task_type="retrieval_document")
    return np.array(result["embedding"])

print("Embedding all examples (one-time cost)...")
pool_embeddings = [(ex, embed(ex[0])) for ex in example_pool]
print(f"Done! {len(pool_embeddings)} examples embedded.\n")

# ── Step 3: For each query, find the most similar examples ──
def find_best_examples(query: str, k: int = 3):
    """Find the k most similar examples to the query."""
    query_emb = embed(query)
    
    # Compute similarity to every example
    scored = []
    for (example, label), emb in pool_embeddings:
        similarity = np.dot(query_emb, emb) / (np.linalg.norm(query_emb) * np.linalg.norm(emb))
        scored.append((similarity, example, label))
    
    # Sort by similarity (highest first) and take top k
    scored.sort(reverse=True)
    return [(ex, label) for _, ex, label in scored[:k]]

# ── Step 4: Build the few-shot prompt with selected examples ──
def dynamic_classify(query: str, k: int = 3) -> str:
    """Classify a query using dynamically selected examples."""
    best_examples = find_best_examples(query, k)
    
    print(f"  Query: \"{query[:60]}...\"")
    print(f"  Selected examples (most similar):")
    for ex, label in best_examples:
        print(f"    → [{label}] \"{ex[:50]}...\"")
    
    prompt = build_few_shot_prompt(
        "Classify the support ticket. Categories: Hardware, Software, Connectivity, Billing, Account. Reply with ONLY the category.",
        best_examples, query, "Ticket", "Category"
    )
    
    result = ask(prompt).strip()
    print(f"  Result: {result}\n")
    return result

# ── Test it! ──
test_queries = [
    "My mouse scroll wheel stopped working",
    "The calendar sync feature doesn't work with Google Calendar",
    "I see an unauthorized charge of $49.99 on my account",
    "Mobile hotspot keeps disconnecting other devices",
]

print("Dynamic Few-Shot Classification\n")
for q in test_queries:
    dynamic_classify(q)


> **What just happened?** For each new query, we: (1) embedded the query into a vector, (2) compared it against all 15 example vectors using cosine similarity, (3) picked the 3 most similar examples, (4) built a few-shot prompt using only those 3. A query about a "mouse scroll wheel" automatically got hardware examples. A query about "unauthorized charge" automatically got billing examples. Different query = different examples = better accuracy. This is how production AI systems work.


### Step 7 · Many-Shot Prompting — 100+ Examples in 2M Token Contexts

Google DeepMind (NeurIPS 2024): hundreds of examples can match fine-tuning.

**`many_shot.py`** · _Block 7 of 10_

MANY-SHOT ICL (Agarwal et al. 2024, NeurIPS)


In [ ]:
# =============================================
# MANY-SHOT ICL (Agarwal et al. 2024, NeurIPS)
# =============================================

# Context windows in 2025-2026:
#   Gemini 2.5/3 Pro:  1-2M tokens
#   Claude 3.5/4:      200K tokens
#   GPT-4o/5:          128K tokens

# With 1M tokens, you can fit 1000+ examples!
# Performance improves LOG-LINEARLY with shot count

print("""
MANY-SHOT KEY FINDINGS:
  • 997-shot translation BEAT Google Translate on Kurdish/Tamil
  • Math (MATH dataset): +35% accuracy with more examples
  • Many-shot can OVERRIDE pretraining biases (few-shot cannot)
  • Diminishing returns vary: some tasks peak at ~125 shots,
    others improve to thousands

TWO NEW PARADIGMS:
  Reinforced ICL:    Use model-generated CoT rationales
                     instead of human examples → 83% on BIG-Bench Hard
  Unsupervised ICL:  Domain questions with NO answers at all
                     → still effective on complex reasoning!

WHEN TO USE:
  Few-shot (3-5):   GPT/Claude with 128-200K context, simple tasks
  Many-shot (50+):  Gemini with 1M+ context, hard classification
  Fine-tuning:      >200K requests, need lowest latency
  
  ⚠️ For reasoning models (o3, GPT-5 reasoning): use FEWER
     examples (1-2). They handle reasoning internally.
""")


> **What just happened?** Many-shot ICL (Google DeepMind, NeurIPS 2024) showed that with 1M+ context windows, hundreds of examples can match fine-tuning on many tasks. 997-shot translation beat Google Translate on low-resource languages. But for reasoning models (o3, GPT-5): use fewer examples (1-2). The new decision tree: model type + context window size + task complexity determines optimal shot count.


### Step 8 · Contrastive Examples & When Few-Shot Hurts

Show what NOT to do. And know when adding examples makes things worse.

**`contrastive.py`** · _Block 8 of 10_

CONTRASTIVE FEW-SHOT — Show good AND bad


In [ ]:
# =============================================
# CONTRASTIVE FEW-SHOT — Show good AND bad
# =============================================

contrastive_prompt = """Write a product description.

PREFERRED example:
  Input: Wireless earbuds, 8hr battery
  Output: "Experience freedom with 8 hours of uninterrupted audio."
  
LESS PREFERRED example:
  Input: Wireless earbuds, 8hr battery
  Output: "These earbuds are wireless and have 8 hour battery life."
  Why worse: Just restates features instead of creating desire.

Now write for: Smart water bottle, tracks hydration
"""

print("""
CONTRASTIVE RESULTS (AAAI 2024):
  +16.0 accuracy on Bamboogle
  +15.2 accuracy on GSM8K math
  Especially effective for style/preference alignment

WHEN FEW-SHOT HURTS (5 failure modes):
  1. OVER-PROMPTING: Too many examples → model copies flaws
  2. WRONG FORMAT: One inconsistent example poisons the whole prompt
  3. BIAS AMPLIFICATION: Unbalanced classes create prediction bias
  4. COMPLEX REASONING: Examples constrain thinking → use zero-shot CoT
  5. CROSS-MODEL FRAGILITY: GPT-optimized examples may fail on Claude

  Rule: If accuracy DROPS after adding examples, REMOVE them.
  More examples ≠ always better.
""")


> **What just happened?** Contrastive examples (showing good AND bad outputs) improve accuracy by 15-16% on hard tasks. But few-shot can also hurt : over-prompting, inconsistent formatting, bias amplification, or constraining reasoning. Rule: if accuracy drops after adding examples, remove them. Always measure before and after.


### Step 9 · Prompt Caching & Token Budgets — Make Examples Nearly Free

Prompt caching changed the economics. Front-load examples, pay once.

**`caching.py`** · _Block 9 of 10_

PROMPT CACHING — Few-shot examples become free


In [ ]:
# =============================================
# PROMPT CACHING — Few-shot examples become free
# =============================================

print("""
PROMPT CACHING ECONOMICS (2025):
  Anthropic: 90% off cached tokens (0.1x base price), 5-min TTL
  OpenAI:    50% off cached tokens (automatic, no code changes)
  Google:    75% off cached tokens (0.25x base price)

ARCHITECTURE PATTERN:
  ┌─────────────────────────────────────┐
  │  CACHED PREFIX (front-load these)   │
  │  ├─ System prompt                   │
  │  ├─ Policies / rules               │
  │  └─ Few-shot examples (3-5)        │
  ├─────────────────────────────────────┤
  │  FRESH SUFFIX (varies per request)  │
  │  └─ User's specific input          │
  └─────────────────────────────────────┘

REAL COST EXAMPLE:
  100K conversations/month, 4KB system + 2KB examples
  Without caching: $50K/month
  With caching:    $15K/month (70% savings!)

TOKEN BUDGETING:
  Each few-shot example ≈ 60-200 tokens
  5 examples ≈ 300-1000 tokens
  Hindi examples cost 2-3x more tokens than English!
  
  Available context = Model limit - (system + examples + query + response)
  Budget examples BEFORE adding them.
""")


> **What just happened?** Prompt caching (Anthropic 90% off, OpenAI 50%, Google 75%) makes few-shot examples nearly free in production. Architecture: front-load stable examples in the cached prefix, put user input in the fresh suffix. Token budgeting: each example costs 60-200 tokens; Hindi examples cost 2-3x more. Always calculate your token budget before adding examples.


### Step 10 · Project: Dynamic Few-Shot Classification System

Combine everything into a production-ready pipeline with accuracy measurement.

**`project_dynamic_fewshot.py`** · _Block 10 of 10_

FULL PROJECT: Dynamic Few-Shot Classifier — - Embeds example pool once


In [ ]:
# =============================================
# FULL PROJECT: Dynamic Few-Shot Classifier
# - Embeds example pool once
# - Selects best examples per query
# - Compares static vs dynamic accuracy
# =============================================

import google.generativeai as genai
import numpy as np
import os, time

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class DynamicFewShotClassifier:
    """Production-ready few-shot classifier with embedding retrieval."""
    
    def __init__(self, task_description, categories, examples):
        self.task = task_description
        self.categories = categories
        self.examples = examples
        self.model = genai.GenerativeModel("gemini-2.0-flash", generation_config={"temperature": 0.1})
        
        # Embed all examples once (cache this in production!)
        print(f"Embedding {len(examples)} examples...")
        self.embedded = []
        for text, label in examples:
            emb = self._embed(text)
            self.embedded.append((text, label, emb))
        print("Ready!\n")
    
    def _embed(self, text):
        result = genai.embed_content(model="models/text-embedding-004", content=text, task_type="retrieval_query")
        return np.array(result["embedding"])
    
    def _find_similar(self, query_emb, k=3):
        scored = []
        for text, label, emb in self.embedded:
            sim = np.dot(query_emb, emb) / (np.linalg.norm(query_emb) * np.linalg.norm(emb))
            scored.append((sim, text, label))
        scored.sort(reverse=True)
        return [(t, l) for _, t, l in scored[:k]]
    
    def classify(self, query, k=3):
        """Classify using dynamically selected examples."""
        query_emb = self._embed(query)
        best = self._find_similar(query_emb, k)
        
        prompt = self.task + "\n\n"
        for ex, label in best:
            prompt += f"Input: {ex}\nCategory: {label}\n\n"
        prompt += f"Input: {query}\nCategory:"
        
        result = self.model.generate_content(prompt).text.strip()
        return result, best
    
    def classify_static(self, query, static_examples):
        """Classify using FIXED examples (for comparison)."""
        prompt = self.task + "\n\n"
        for ex, label in static_examples:
            prompt += f"Input: {ex}\nCategory: {label}\n\n"
        prompt += f"Input: {query}\nCategory:"
        return self.model.generate_content(prompt).text.strip()

# ── Create the classifier ──
classifier = DynamicFewShotClassifier(
    task_description="Classify support tickets. Reply with ONLY one category: Hardware, Software, Connectivity, Billing, Account.",
    categories=["Hardware", "Software", "Connectivity", "Billing", "Account"],
    examples=example_pool,  # our 15 examples from Part 5
)

# ── Test: Static vs Dynamic ──
static_examples = example_pool[:3]  # always use first 3 (Hardware, Software, Connectivity)

test_set = [
    ("My keyboard makes a clicking sound but doesn't type", "Hardware"),
    ("The search function returns wrong results", "Software"),
    ("4G data isn't working after traveling abroad", "Connectivity"),
    ("I was charged for a plan I never signed up for", "Billing"),
    ("I need to delete my account permanently", "Account"),
    ("USB port stopped recognizing any device", "Hardware"),
    ("The notification settings reset after every update", "Software"),
    ("Can I transfer my subscription to another person?", "Billing"),
]

print("Comparing Static vs Dynamic Few-Shot:\n")
print(f"{'Query':<45} {'Static':>10} {'Dynamic':>10} {'Expected':>10}")
print("─" * 80)

static_correct = 0
dynamic_correct = 0

for query, expected in test_set:
    static_ans = classifier.classify_static(query, static_examples)
    dynamic_ans, _ = classifier.classify(query, k=3)
    
    s_ok = expected.lower() in static_ans.lower()
    d_ok = expected.lower() in dynamic_ans.lower()
    static_correct += s_ok
    dynamic_correct += d_ok
    
    print(f"  {query[:43]:<45} {'✅' if s_ok else '❌':>4}{static_ans:>8} {'✅' if d_ok else '❌':>4}{dynamic_ans:>8} {expected:>10}")

print(f"\n{'─' * 80}")
print(f"Static accuracy:  {static_correct}/{len(test_set)} ({static_correct/len(test_set):.0%})")
print(f"Dynamic accuracy: {dynamic_correct}/{len(test_set)} ({dynamic_correct/len(test_set):.0%})")

if dynamic_correct > static_correct:
    print(f"\nDynamic wins by {dynamic_correct - static_correct} points!")

print("""
Why dynamic is better:
  Static always shows Hardware + Software + Connectivity examples.
  When the query is about "Billing" or "Account", the model has 
  NO relevant examples → guesses wrong.
  
  Dynamic finds the 3 examples CLOSEST to the query.
  Billing query → gets billing examples → nails it.
""")


> **What just happened?** We built a complete DynamicFewShotClassifier class that: (1) embeds all examples once at startup, (2) for each query, finds the 3 most similar examples, (3) builds a prompt with those examples, (4) classifies. Then we proved it works better than static few-shot by testing both on 8 queries. Static gets ~60-75% (because it misses categories). Dynamic gets ~90-100% (because it always finds relevant examples). This is a real production pattern used by companies for customer support, content moderation, and document classification.


---

## Wrap-up

You've walked through all 10 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
